# Multi-Agent Workflows: Orchestration, Evaluation, and Prompt Caching

**MRS Spring 2026 — Tutorial MT01: Deploying Agentic AI in Materials Characterization Workflows**

A single agent with a long tool list eventually starts:
- Forgetting which tool does what
- Losing the thread of complex reasoning
- Costing too much in tokens because every step ships the full system prompt

This notebook shows three patterns that go together in production scientific
agents:

1. **Multi-agent orchestration** — a *planner* dispatches work to specialist
   sub-agents (literature, data analysis), then synthesizes the answer.
2. **Evaluation** — a tiny eval harness with traces, so you can tell whether
   "your agent works" is actually true.
3. **Prompt caching** — Anthropic's caching feature, with a measured cost
   comparison. For long system prompts (which sub-agents always have), this
   is a 5-10x cost reduction.

We will reuse the synthesis-extraction and RAG ideas from earlier notebooks,
plus the XRD analysis use case, to ground everything in materials science.

---
## Setup

In [ ]:
# !pip install anthropic numpy

In [ ]:
import os, getpass, json, time, textwrap, numpy as np
from dataclasses import dataclass, field
from typing import Any

if "ANTHROPIC_API_KEY" not in os.environ:
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API key: ")

import anthropic
client = anthropic.Anthropic()

# Sonnet for the planner (good reasoning), Haiku for the specialists (fast & cheap).
PLANNER_MODEL    = "claude-sonnet-4-6"
SPECIALIST_MODEL = "claude-haiku-4-5-20251001" 

---
## Part 1: Pattern overview

The pattern we'll build:

```
                    ┌──────────────┐
   user question -> │   Planner    │ ──── decides which specialists to call
                    │  (Sonnet)    │ ──── synthesises final answer
                    └──┬────────┬──┘
                       │        │
              ┌────────▼┐    ┌──▼────────┐
              │Literature│    │Analysis   │
              │ agent    │    │ agent     │
              │ (Haiku)  │    │ (Haiku)   │
              └──────────┘    └───────────┘
```

The planner doesn't see the specialists' raw tools — it sees them as **two
high-level tools** (`ask_literature_agent`, `ask_analysis_agent`). Each
specialist has its own focused system prompt, smaller tool set, and cheaper
model.

This is the same pattern Anthropic's research agents use, and the same pattern
Claude Code uses to spawn sub-agents.

---
## Part 2: The specialists

Each specialist is a function: question in, answer out. Internally it runs
its own tool-use loop with its own focused tools.

In [ ]:
# --- Toy "literature" backend (would be a real RAG index in production) ---
LITERATURE = {
    "BaTiO3": "BaTiO3 is a tetragonal perovskite (P4mm). Curie temperature ~120 C. "
              "Sol-gel synthesis typically calcines at 700-800 C.",
    "SrTiO3": "SrTiO3 is a cubic perovskite (Pm-3m), a=3.905 A. Bandgap 3.25 eV. "
              "Common substrate; hydrothermally grown at 200 C from SrCl2 + TiO2.",
    "LiFePO4": "LiFePO4 olivine-structure cathode (Pnma). Solid-state synthesis at "
               "~700 C in inert atmosphere. Theoretical capacity 170 mAh/g.",
}

LITERATURE_TOOLS = [{
    "name": "search_literature",
    "description": "Look up a known material's reference properties.",
    "input_schema": {"type": "object",
                     "properties": {"formula": {"type": "string"}},
                     "required": ["formula"]},
}]

def run_literature_agent(question: str) -> str:
    sys_prompt = ("You are a materials-science literature specialist. "
                  "Use the search_literature tool to ground every claim. "
                  "Return a concise factual answer.")
    messages = [{"role": "user", "content": question}]
    for _ in range(4):
        resp = client.messages.create(
            model=SPECIALIST_MODEL, max_tokens=512,
            system=sys_prompt, tools=LITERATURE_TOOLS, messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for tu in tool_uses:
            f = tu.input.get("formula", "")
            results.append({
                "type": "tool_result", "tool_use_id": tu.id,
                "content": LITERATURE.get(f, f"No entry for {f}"),
            })
        messages.append({"role": "user", "content": results})
    return "[literature agent: max turns]"

print(run_literature_agent("What is the Curie temperature of BaTiO3?"))

In [ ]:
# --- "Data analysis" specialist: peak picking on a 1-D pattern ---
from scipy.signal import find_peaks

ANALYSIS_TOOLS = [{
    "name": "find_peaks_in_pattern",
    "description": "Find peaks in a 1-D XRD pattern.",
    "input_schema": {
        "type": "object",
        "properties": {
            "two_theta":  {"type": "array", "items": {"type": "number"}},
            "intensity":  {"type": "array", "items": {"type": "number"}},
        },
        "required": ["two_theta", "intensity"],
    },
}]

def run_analysis_agent(question: str, attached_data: dict) -> str:
    sys_prompt = ("You are an XRD data-analysis specialist. Use the available "
                  "tools rather than guessing. Report peaks as a numeric list.")
    user_msg = (f"{question}\n\n"
                f"Attached data: arrays of length {len(attached_data['two_theta'])}.")
    messages = [{"role": "user", "content": user_msg}]
    for _ in range(4):
        resp = client.messages.create(
            model=SPECIALIST_MODEL, max_tokens=512,
            system=sys_prompt, tools=ANALYSIS_TOOLS, messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            return "".join(b.text for b in resp.content if b.type == "text")
        results = []
        for tu in tool_uses:
            x = np.asarray(attached_data["two_theta"])
            y = np.asarray(attached_data["intensity"])
            idx, _ = find_peaks(y, height=0.05*y.max(), distance=5)
            results.append({
                "type": "tool_result", "tool_use_id": tu.id,
                "content": json.dumps({"peak_2theta": [float(x[i]) for i in idx]}),
            })
        messages.append({"role": "user", "content": results})
    return "[analysis agent: max turns]"

# quick sanity check
xs = np.linspace(20, 60, 4001)
ys = np.exp(-((xs-22.0)/0.15)**2) + 0.7*np.exp(-((xs-31.5)/0.15)**2)
print(run_analysis_agent("Where are the peaks?", {"two_theta": xs.tolist(), "intensity": ys.tolist()}))

---
## Part 3: The planner

The planner sees the two specialists as tools. It picks which to call, in
which order, and synthesises the final answer.

In [ ]:
PLANNER_TOOLS = [
    {
        "name": "ask_literature_agent",
        "description": "Ask the literature specialist a question about known materials.",
        "input_schema": {"type": "object",
                         "properties": {"question": {"type": "string"}},
                         "required": ["question"]},
    },
    {
        "name": "ask_analysis_agent",
        "description": "Ask the data-analysis specialist to operate on the attached XRD data.",
        "input_schema": {"type": "object",
                         "properties": {"question": {"type": "string"}},
                         "required": ["question"]},
    },
]

@dataclass
class Trace:
    """A simple trajectory log we'll use for evaluation later."""
    user_question: str
    steps: list = field(default_factory=list)
    final_answer: str = ""
    cost_usd: float = 0.0

def run_planner(user_question: str, attached_data: dict | None = None) -> Trace:
    trace = Trace(user_question=user_question)
    sys_prompt = ("You are the lead scientific orchestrator. You have access to two "
                  "specialist agents. For factual questions about known materials, "
                  "call ask_literature_agent. For numerical analysis of attached data, "
                  "call ask_analysis_agent. Combine their answers into a clear final "
                  "response.")
    messages = [{"role": "user", "content": user_question}]
    for _ in range(6):
        resp = client.messages.create(
            model=PLANNER_MODEL, max_tokens=1024,
            system=sys_prompt, tools=PLANNER_TOOLS, messages=messages,
        )
        messages.append({"role": "assistant", "content": resp.content})
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        if not tool_uses:
            trace.final_answer = "".join(b.text for b in resp.content if b.type == "text")
            return trace
        results = []
        for tu in tool_uses:
            if tu.name == "ask_literature_agent":
                ans = run_literature_agent(tu.input["question"])
            elif tu.name == "ask_analysis_agent":
                ans = run_analysis_agent(tu.input["question"], attached_data or {})
            else:
                ans = f"unknown tool: {tu.name}"
            trace.steps.append({"tool": tu.name, "input": tu.input, "output": ans})
            results.append({"type": "tool_result", "tool_use_id": tu.id, "content": ans})
        messages.append({"role": "user", "content": results})
    trace.final_answer = "[planner: max turns]"
    return trace

# Demo run
xs = np.linspace(20, 60, 4001)
ys = np.exp(-((xs-22.7)/0.15)**2) + 0.7*np.exp(-((xs-32.4)/0.15)**2) + 0.4*np.exp(-((xs-39.9)/0.15)**2)
data = {"two_theta": xs.tolist(), "intensity": ys.tolist()}

trace = run_planner(
    "I have an XRD pattern attached. Find the peaks and tell me which "
    "well-known cubic perovskite they're consistent with, including its space group.",
    attached_data=data,
)

print("=== Planner steps ===")
for s in trace.steps:
    print(f"\n* {s['tool']}({s['input']})\n  -> {s['output'][:200]}...")
print("\n=== Final answer ===\n")
print(trace.final_answer)

Notice the **separation of concerns**: the planner doesn't know how peak
finding works; the analysis specialist doesn't know what perovskites are; the
literature specialist doesn't see the raw arrays. Each is a focused expert.

---
## Part 4: Evaluation — does it actually work?

"My agent works" is the most over-claimed sentence in agentic AI. The only
honest answer comes from a fixed eval set with explicit success criteria.

We'll build a tiny evaluator that:
- runs the planner over a list of known-answer questions,
- inspects each trace,
- scores correctness (with a simple LLM-as-judge).

In [ ]:
EVAL_SET = [
    {
        "id": "eval-01",
        "question": "What is the Curie temperature of BaTiO3?",
        "must_contain": ["120", "Curie"],          # cheap substring check
    },
    {
        "id": "eval-02",
        "question": "What space group does SrTiO3 have?",
        "must_contain": ["Pm-3m"],
    },
    {
        "id": "eval-03",
        "question": "List two well-known perovskite oxides.",
        "must_contain": ["BaTiO3", "SrTiO3"],
    },
]

def evaluate_run(trace: Trace, must_contain: list[str]) -> dict:
    answer = trace.final_answer
    hits = [s for s in must_contain if s.lower() in answer.lower()]
    return {
        "passed": len(hits) == len(must_contain),
        "hits":   hits,
        "missed": [s for s in must_contain if s not in hits],
        "n_steps": len(trace.steps),
    }

results = []
for case in EVAL_SET:
    tr = run_planner(case["question"])
    score = evaluate_run(tr, case["must_contain"])
    results.append({"id": case["id"], **score})
    print(f"{case['id']}: {'PASS' if score['passed'] else 'FAIL'} "
          f"(steps={score['n_steps']}, missed={score['missed']})")

passed = sum(r["passed"] for r in results)
print(f"\nOverall: {passed}/{len(results)} pass")

The substring check is intentionally crude. In a real eval pipeline you'd
add:
- An **LLM judge** (a separate Claude call rating each answer 1-5).
- **Trajectory checks** ("did it actually call the literature agent?").
- **Regression** detection — re-run on every prompt change and alert on drops.

The point of having *any* eval set, no matter how small, is that it turns
"my agent feels good" into a number. Numbers can be trended, optimised,
and CI'd.

---
## Part 5: Prompt caching — the multi-agent cost killer

Each time we call a specialist, we ship its (long) system prompt. With three
specialists and 6 planner turns per question, that's a lot of repeated tokens.

Anthropic's **prompt caching** lets us mark stable prefixes (system prompt,
tool definitions, large documents) and pay for them *once* per 5-minute
window. Cached reads are ~10x cheaper than fresh tokens.

To use it, add `cache_control={"type": "ephemeral"}` to the **last block**
you want included in the cache prefix.

In [ ]:
LONG_SYSTEM = (
    "You are a materials-science research assistant. You have deep knowledge of "
    "crystallography, synthesis, characterisation, and battery/photovoltaic "
    "materials. " * 200  # ~6k tokens — well above the 1024-token min for Sonnet
)

# Anthropic's prompt cache has a minimum cacheable size: 1024 tokens for
# Sonnet, 2048 for Haiku. If your cached portion is below the threshold the
# API silently skips caching (you'll see cache_*_tokens stay at 0). The
# padding above keeps us safely above that for both model families.

def call_with_caching(question: str, use_cache: bool):
    system_blocks = [{"type": "text", "text": LONG_SYSTEM}]
    if use_cache:
        system_blocks[0]["cache_control"] = {"type": "ephemeral"}
    resp = client.messages.create(
        model=PLANNER_MODEL, max_tokens=200,
        system=system_blocks,
        messages=[{"role": "user", "content": question}],
    )
    return {
        "input_tokens":             resp.usage.input_tokens,
        "cache_creation_tokens":    getattr(resp.usage, "cache_creation_input_tokens", 0),
        "cache_read_tokens":        getattr(resp.usage, "cache_read_input_tokens", 0),
        "output_tokens":            resp.usage.output_tokens,
    }

print("--- Run 1 (no caching) ---")
print(call_with_caching("Name a tetragonal perovskite.", use_cache=False))

print("\n--- Run 2 (caching enabled, cold) ---")
print(call_with_caching("Name a tetragonal perovskite.", use_cache=True))

print("\n--- Run 3 (caching enabled, warm — within 5 min) ---")
print(call_with_caching("Name a cubic perovskite.", use_cache=True))

**What you should see:**
- Run 1: all input tokens billed at full rate.
- Run 2: most tokens reported as `cache_creation_tokens` (small premium over
  base, charged once).
- Run 3: most tokens reported as `cache_read_tokens` (~10% of base price).

**Rule of thumb:** anything over ~1000 tokens that you reuse more than twice
in 5 minutes should be cached. For multi-agent systems where every specialist
call repeats a system prompt, this is a no-brainer.

---
## Putting it together

A production-quality scientific agent typically combines all three:

| Concern | Lever |
|---|---|
| Reasoning quality on complex tasks | Multi-agent orchestration with focused specialists |
| Trustworthiness | Eval set with traces, run on every prompt change |
| Cost / latency | Prompt caching on the stable prefixes |

You now have all three patterns in one notebook. The next notebook shows how
to organise the same orchestration logic using a *graph* abstraction
(LangGraph) — useful when your workflow has branching, retries, or long-lived
state.